<a href="https://colab.research.google.com/github/dharamdevm/mrdevsacademy/blob/main/quickstarts/Get_started_managed_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 Google LLC.

In [104]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Agents API: Build managed agents with the Interactions API

<a class="tfo-notebook-buttons" target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_managed_agents.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The [Interactions API](https://ai.google.dev/gemini-api/docs/interactions) provides a unified interface for working with Gemini models and agents. The [Getting Started notebook](./Get_started_interactions_api.ipynb) covers how to use it with standard Gemini **models** for text generation, multi-turn conversations, and tool use.

This notebook focuses on something different: **managed agents** with the `antigravity-preview-05-2026` agent.

### `agent=` vs `model=`

When you call the Interactions API, you choose between two modes:

| Parameter | What runs | Best for |
|-----------|-----------|----------|
| `model="gemini-..."` | A standard Gemini model | Text generation, structured output, function calling |
| `agent="antigravity-preview-05-2026"` | A **managed agent** in a sandboxed Linux environment | Autonomous tasks: code execution, web research, file management |

With `model=`, you get a stateless LLM call (see the [Getting Started notebook](./Get_started_interactions_api.ipynb)). With `agent=`, you spin up an autonomous agent that can **reason, plan, write and execute code, browse the web, and manage files** — all inside a secure sandbox, without you writing any orchestration logic.

This notebook walks you through the agent mode step by step:

1. **Simple questions** — use the agent like an LLM (it works, but it's overkill!)
2. **Multi-turn conversations** — persistent sandbox = built-in memory
3. **Using tools** — code execution, web search, file operations
4. **Loading data into the sandbox** — inject files before the agent starts
5. **Creating reusable custom agents** — bundle instructions, skills, and environment

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [105]:
%pip install -U -q "google-genai>=2.9.0"

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key or you aren't sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [106]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK, now you only need to initialize a client with you API key.

In [107]:
import uuid
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=GEMINI_API_KEY)

# The default managed agent.
AGENT = "antigravity-preview-05-2026"

# Generate a unique suffix for this notebook session to prevent agent ID conflicts
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

print("Client ready!")

Client ready!


## 1. Simple questions — the agent as an LLM

The simplest way to use a managed agent is to ask it a question, just like you'd call a standard Gemini model. Pass `agent="antigravity-preview-05-2026"` and `environment="remote"` to create a fresh Linux sandbox for the agent.

This works, but it's a bit like driving a Formula 1 car to the grocery store — the agent has code execution, web search, and file management capabilities that are all sitting idle for a simple factual question.

In [108]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What is the capital of France?",
    environment="remote",
)

Markdown(interaction.output_text)

The capital of France is Paris.

In [109]:
# The response also includes metadata about the agent's sandbox.
print(f"Status:         {interaction.status}")
print(f"Interaction ID: {interaction.id}")
print(f"Environment ID: {interaction.environment_id}")

Status:         completed
Interaction ID: v1_ChdGc0YtYW9Yc0RfblBfdU1QZ3ZlWnNRZxIXRnNGLWFvWHNEX25QX3VNUGd2ZVpzUWc
Environment ID: ce01565b9e45a085808ec5e61d54fc68


Notice the `environment_id` in the response. That's the agent's persistent Linux sandbox. Even for this simple question, a full container was provisioned. Let's make use of that persistence next.

## 2. Multi-turn conversations

Since each agent runs in a persistent sandbox, you can **continue where you left off** by reusing the `environment_id` and linking turns with `previous_interaction_id`.

This is fundamentally different from stateless `model=` calls. The agent has a true *persistent environment* — files it creates stick around, packages it installs remain available, and conversation context is preserved.

In [110]:
# Turn 1: Introduce yourself.
turn1 = client.interactions.create(
    agent=AGENT,
    input="Hi! My name is Alice and I'm a software engineer. Remember that in a knowledge.md doc.",
    environment= "remote",
)

Markdown(f"**Turn 1:** {turn1.output_text}")

**Turn 1:** Nice to meet you, Alice! I have created `knowledge.md` and recorded that you are a software engineer. Let me know how I can help you today!

In [111]:
# Turn 2: Ask whether the agent remembers.
# Pass environment_id and previous_interaction_id to continue the conversation.
turn2 = client.interactions.create(
    agent=AGENT,
    input="what's my name and what do I do?",
    environment= turn1.environment_id,
)

Markdown(f"**Turn 2:** {turn2.output_text}")

**Turn 2:** I don't have access to your name or what you do, as you haven't shared that information with me yet. 

Feel free to introduce yourself or let me know what task or project you'd like to work on!

The agent remembered across turns because you passed `environment` with the previous environment ID — same sandbox, which means same files.

You could have achieved the same result using `previous_interaction_id` to keep the history of the previous conversation, but that would not have showcased the environement specificities.

This is how you build stateful, multi-turn workflows. See the [Getting Started notebook](./Get_started_interactions_api.ipynb) for `model=`-based multi-turn using `previous_interaction_id` alone (without environments).

## 3. Using tools — where the agent shines

This is where managed agents go beyond a standard chat model. The antigravity-preview-05-2026 agent has **built-in tools** it uses autonomously — you don't declare them, just describe your goal and the agent figures out what to use.

| Tool | Description |
|------|-------------|
| `bash` | Execute shell commands in the sandbox |
| `google_search` | Search the web for current information |
| `url_context` | Fetch and extract text from URLs |
| `write_file` | Create or overwrite files in the sandbox |
| `read_file` | Read file contents from the sandbox |
| `list_files` | List directory contents |
| `delete_file` | Remove files from the sandbox |

For the standard `model=`-based tools (Google Search grounding, code execution, function calling), see the [Getting Started notebook](./Get_started_interactions_api.ipynb) and the dedicated tool notebooks:
- [Code Execution](./Code_Execution.ipynb)
- [Search Grounding](./Search_Grounding.ipynb)
- [Function Calling](./Function_calling.ipynb)

### Code execution

Ask a computational question and the agent will write code, run it in its sandbox, and return the verified result.

In [113]:
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Write a Python script that computes the first 20 Fibonacci numbers. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

Here is the Python script (`fibonacci.py`) to compute the first 20 Fibonacci numbers:

```python
def fibonacci(n):
    fib_series = []
    a, b = 0, 1
    for _ in range(n):
        fib_series.append(a)
        a, b = b, a + b
    return fib_series

if __name__ == "__main__":
    n = 20
    fib_numbers = fibonacci(n)
    print(f"The first {n} Fibonacci numbers are:")
    print(fib_numbers)
```

### Execution Output:

```
The first 20 Fibonacci numbers are:
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]
```

### Inspecting steps — what the agent actually did

The `steps` field in the response shows the agent's reasoning chain: its thoughts, tool calls, tool results, and final output. This is useful for debugging and understanding the agent's behavior.

In [ ]:
# Inspect the steps from the Fibonacci interaction above.
for i, step in enumerate(interaction.steps):
    step_type = step.type
    print(f"--- Step {i} [{step_type}] ---")

    # Tool call steps show which tool was invoked and with what arguments.
    if hasattr(step, "name") and step.name:
        print(f"  Tool: {step.name}")
        if hasattr(step, "arguments"):
            args_str = str(step.arguments)[:300]
            print(f"  Args: {args_str}")

    # Content steps contain the agent's text output.
    if hasattr(step, "content") and step.content:
        for c in step.content:
            if hasattr(c, "text"):
                print(f"  Text: {c.text[:300]}")
    print()

### Web search

The agent can search the web autonomously when it needs up-to-date information.

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What were the top 3 news stories about Google this week? Summarize them briefly.",
    environment="remote",
)

Markdown(interaction.output_text)

### File operations

The agent can create, read, and manage files in its sandbox. Files persist within the environment across turns.

In [ ]:
# Ask the agent to create a file, run it, and show results.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a Python file called 'analysis.py' that generates 50 random numbers, "
        "computes mean, median, and standard deviation, then prints the results. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

## 4. Loading data into the agent's sandbox

You can inject files into the agent's environment **before it starts** using `sources`. This is how you provide data, configuration, or code for the agent to work with.

| Source type | Description | Best for |
|------------|-------------|----------|
| `inline` | Embed content directly (max 75 KB) | Config files, small scripts |
| `gcs` | Load from Google Cloud Storage | Large datasets |
| `repository` | Load from GitHub | Code repositories |

In [ ]:
# Inject a CSV file inline and ask the agent to analyze it.
csv_data = """name,age,city,score
              Alice,28,Paris,92
              Bob,35,London,87
              Charlie,42,Berlin,95
              Diana,31,Tokyo,88
              Eve,26,Sydney,91"""

interaction = client.interactions.create(
    agent=AGENT,
    input="Read the file data.csv, analyze it, and tell me who scored the highest.",
    environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "content": csv_data,
                "target": "/workspace/data.csv",
            }
        ],
    },
)

Markdown(interaction.output_text)

You can also load from other sources:

```python
# From Google Cloud Storage
{"type": "gcs", "source": "gs://my-bucket/data/", "target": "/workspace/data/"}

# From a GitHub repository
{"type": "repository", "source": "https://github.com/user/repo", "target": "/workspace/repo/"}
```

You can combine multiple sources in a single request — the agent will have access to all of them at startup.

**Pro tip:** You can use that to add skills to you agent, as you'll see next.

## 5. Creating reusable custom agents

So far, every interaction has used the base `antigravity-preview-05-2026` agent with inline instructions. Once you've found a setup that works well, you can **persist it into a named custom agent** that bundles:

- **Instructions** — system prompt that defines the agent's behavior
- **Environment** — pre-configured sandbox with files and sources
- **Skills** — `SKILL.md` files that teach the agent specialized capabilities

This is the recommended workflow:
1. **Prototype** with `agent="antigravity-preview-05-2026"` — iterate on instructions, sources, and prompts
2. **Create** a named agent via the `/agents` endpoint
3. **Invoke** your agent by name from any client

### Creating a custom agent

In [ ]:
# Create a custom data analysis agent using the SDK.
my_agent = client.agents.create(
    id=f"my-data-analyst-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a data analysis assistant. "
        "Always write Python code using pandas to answer questions. "
        "Show your code and output clearly. "
        "When creating visualizations, save them as PNG files."
    ),
    base_environment={
        "type": "remote",
    },
)

print(f"✓ Agent created: {my_agent.id}")

### Using a custom agent

Once created, invoke your agent by name. It will follow its instructions automatically.

In [ ]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-data-analyst-{UNIQUE_SUFFIX}",
    input=(
        "Generate a sample dataset of 100 sales records with columns: "
        "product, region, revenue, quantity. "
        "Find the top 5 products by total revenue and show the analysis."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

### Creating an agent with pre-loaded data

You can define the agent's environment with sources, so data is ready before the agent starts:

In [ ]:
# Create an agent with GCS sources pre-loaded using the SDK.
my_slides_agent = client.agents.create(
    id=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a software engineer speciliazed in the Gemini API. "
        "Use the skills available in /.agents/skills/ to create amazing apps."
    ),
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "repository",
                "source": "https://github.com/google-gemini/gemini-skills",
                "target": "/.agents/skills",
            }
        ],
    },
)

print(f"✓ Agent created: {my_slides_agent.id}")

In [ ]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    input="Tell me what you can do with your skills?",
    environment="remote",
)

Markdown(interaction.output_text)

### Forking from an existing environment

If you've already set up a sandbox you like (installed packages, created files, etc.), you can fork it into a new agent using the `environment_id` from a previous interaction:

```python
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT,
    system_instruction="Your custom instructions here.",
    base_environment={"env_id": "YOUR_ENVIRONMENT_ID"},
)
```

This captures the exact state of that sandbox — all installed packages, files, and configuration.

In [ ]:
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent="my-gemini-api-agent",
    system_instruction="I want all your apps to use the Live API",
    base_environment={"env_id": interaction.environment_id},
)
print(f"✓ Agent forked: {my_forked_agent.id}")

### Managing agents (CRUD)

The `/agents` endpoint supports full lifecycle management:

In [ ]:
# List all your agents.
print("Your agents:")
for agent in client.agents.list().agents:
    print(f"- {agent.id}")

# Get a specific agent's details.
agent = client.agents.get(id="my-data-analyst")
print(f"\nAgent details for {agent.id}:")
print(f"Base agent: {agent.base_agent}")
print(f"System instruction: {agent.system_instruction}")

In [ ]:
# Clean up: delete the agents you created.
for agent_name in ["my-data-analyst", "my-forked-agent", "my-gemini-api-agent"]:
    try:
        client.agents.delete(id=agent_name)
        print(f"✓ Deleted {agent_name}")
    except Exception as e:
        print(f"  Failed to delete {agent_name}: {e}")

### Agent directory structure

Behind the API, an agent is defined by a simple set of files. This is what gets deployed when you create one:

```
my-agent/
├── agent.yaml       # Configuration: base agent, tools, environment
├── AGENTS.md        # System instructions (loaded automatically)
├── skills/          # Custom SKILL.md files that extend capabilities
└── workspace/       # Files seeded into the remote sandbox at startup
```

- **`agent.yaml`** maps directly to the `/agents` API resource
- **`AGENTS.md`** provides system instructions — automatically loaded by the harness
- **`skills/`** contains specialized `SKILL.md` files the agent discovers and uses
- **`workspace/`** files are injected into the sandbox at startup

This file-based structure makes agents easy to version-control, share, and iterate on. Check the [documentation](https://ai.google.dev/gemini-api/docs/custom-agents#file-based_customization) for more details.

## 6. Streaming

For longer tasks, enable streaming with `stream=True` to get real-time updates as the agent works. Instead of waiting for the complete response, you receive a stream of **Server-Sent Events (SSE)** that let you show progress to the user.

### Event types

The stream delivers events that tell you what the agent is doing:

| Event type | Meaning | What to do |
|------------|---------|------------|
| `interaction.created` | The interaction was created | Store the `id` for later reference |
| `interaction.status_update` | Status changed (e.g., `in_progress`) | Update UI status indicator |
| `step.start` | A new step began (thinking, tool call, output) | Show a loading indicator |
| `step.delta` | Incremental content — a chunk of text, thought, or tool output | **Append to display** — this is the main content |
| `step.stop` | A step completed | Hide loading indicator |
| `interaction.completed` | The agent finished all work | Finalize the UI |

The `step.delta` events are where the content lives. Each delta has a `type` (e.g., `text`, `thought`, `function_call`, `function_result`) and content you can render incrementally.

In [ ]:
# Stream a response and collect the text as it arrives.
stream = client.interactions.create(
    agent=AGENT,
    input="Write a short poem about the ocean.",
    stream=True,
    environment="remote",
)

collected_text = []

for event in stream:
    # Show the event type so you can see the lifecycle.
    if event.event_type in ("interaction.created", "step.start", "step.stop", "interaction.completed"):
        print(f"[{event.event_type}]")

    # step.delta events carry the actual content.
    elif event.event_type == "step.delta":
        delta = event.delta
        if hasattr(delta, "text") and delta.text:
            print(delta.text, end="", flush=True)
            collected_text.append(delta.text)

print(f"\n\n--- Collected {len(collected_text)} text chunks ---")

## 7. Advanced features

### Network configuration

By default, the agent's sandbox has unrestricted outbound network access. You can control this with the `network` field:
- **Allowlist specific domains** — only requests to listed domains are permitted
- **Inject credentials** — automatically add headers (API keys, tokens) to outbound requests
- **Disable network** — set `network: "disabled"` to block all outbound traffic

In [ ]:
# Allow the agent to call only the Gemini API, with an auto-injected API key.
interaction = client.interactions.create(
    agent=AGENT,
    input="Use curl to call the Gemini API and list available models. Show the first 3.",
    environment={
        "type": "remote",
        "network": {
            "allowlist": [
                {
                    "domain": "generativelanguage.googleapis.com",
                    "transform": [{"x-goog-api-key": GEMINI_API_KEY}],
                },
            ]
        },
    },
)

Markdown(interaction.output_text)

### Download environment snapshots

You can download all the files the agent created or modified as a tar archive. This lets you retrieve the agent's work products — code, data, reports — from the sandbox.

In [ ]:
import subprocess
import tarfile
import os

# Create an interaction where the agent produces files.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a directory called 'project' with a README.md and a hello.py script. "
        "List the files you created."
    ),
    environment="remote",
)

env_id = interaction.environment_id
print(f"Environment ID: {env_id}")

Markdown(interaction.output_text)

In [ ]:
# Download the environment snapshot.
download_url = (
    f"https://generativelanguage.googleapis.com/v1beta/"
    f"files/environment-{env_id}:download?alt=media"
)

result = subprocess.run(
    ["curl", "-L", "-s", "-o", "snapshot.tar",
     "-H", f"x-goog-api-key: {GEMINI_API_KEY}",
     download_url],
    capture_output=True, text=True,
)

if os.path.exists("snapshot.tar") and os.path.getsize("snapshot.tar") > 0:
    with tarfile.open("snapshot.tar") as tar:
        print("Files in snapshot:")
        for member in tar.getmembers():
            print(f"  {member.name} ({member.size} bytes)")
else:
    print("Snapshot not available (environment may have expired).")

## Next steps

You've walked through the core capabilities of managed agents:

1. ✅ **Simple Q&A** — the agent can answer questions like an LLM
2. ✅ **Multi-turn** — persistent sandbox enables stateful conversations
3. ✅ **Built-in tools** — code execution, web search, file management
4. ✅ **Data loading** — inject files via inline, GCS, or GitHub sources
5. ✅ **Custom agents** — reusable configurations with instructions, skills, and environment
6. ✅ **Streaming** — real-time updates as the agent works
7. ✅ **Advanced** — network control and environment snapshots

### Learn more

- **[Getting Started notebook](./Get_started_interactions_api.ipynb)** — the `model=`-based Interactions API for standard generation, multi-turn, and tools
- **[Managed Agents documentation](https://ai.google.dev/gemini-api/docs/eap/gemini-agents/gemini-agents)** — full reference for the agent API
- **[Code Execution](./Code_Execution.ipynb)** — model-based code execution
- **[Search Grounding](./Search_Grounding.ipynb)** — model-based web search
- **[Function Calling](./Function_calling.ipynb)** — custom function declarations

## Resources

In [ ]:
pip install ngrok

RuPay replace to RazorPay

In [ ]:
import os

html_files = [f for f in os.listdir('.') if f.endswith(('.html', '.htm'))]

if html_files:
    print('Found HTML files:')
    for html_file in html_files:
        print(f'- {html_file}')
else:
    print('No HTML files found in the current directory.')

In [ ]:
import os

# To list files in the parent directory (one level up from current)
parent_directory_path = '..'

print(f"Files in '{parent_directory_path}':")
try:
    files_in_parent = os.listdir(parent_directory_path)
    for file_name in files_in_parent:
        print(f'- {file_name}')
except FileNotFoundError:
    print(f"The directory '{parent_directory_path}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

If you want to specify an absolute path or another relative path, simply replace `'..'` with the desired directory path (e.g., `'/content/mydir'` or `'./subdir'`).

In [ ]:
import requests

# IMPORTANT: Replace with the actual URL of your login page
login_page_url = 'https://example.com/login'

try:
    response = requests.get(login_page_url)
    print(f"Status Code for {login_page_url}: {response.status_code}")
    if response.status_code == 200:
        print("The login page is likely accessible.")
    else:
        print("The login page returned a non-200 status code. It might not be accessible as expected or requires authentication.")
except requests.exceptions.RequestException as e:
    print(f"An error occurred while trying to access the login page: {e}")

A status code of `200` generally means the page loaded successfully. Other codes like `404` (Not Found) or `500` (Server Error) would indicate issues.

In [ ]:
import os

html_directory_path = '/content/stitch_talkie_talkie_extracted/stitch_talkie_talkie/account_activity/'

print(f"Searching for HTML files in '{html_directory_path}':")
try:
    html_files_in_dir = [f for f in os.listdir(html_directory_path) if f.endswith(('.html', '.htm'))]

    if html_files_in_dir:
        print('Found HTML files:')
        for html_file in html_files_in_dir:
            print(f'- {html_file}')
    else:
        print('No HTML files found in the specified directory.')
except FileNotFoundError:
    print(f"The directory '{html_directory_path}' was not found. Please ensure the path is correct.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import os
from bs4 import BeautifulSoup

html_file_path = '/content/stitch_talkie_talkie_extracted/stitch_talkie_talkie/account_activity/code.html'

print(f"Reading HTML file: {html_file_path}")

try:
    with open(html_file_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    soup = BeautifulSoup(html_content, 'html.parser')

    print("\nExtracting 'href' attributes from <a> tags:")
    a_tags = soup.find_all('a')

    if a_tags:
        for i, tag in enumerate(a_tags):
            href = tag.get('href')
            if href:
                print(f"  -{i+1}: {href}")
            else:
                print(f"  -{i+1}: <a> tag found but no 'href' attribute.")
    else:
        print("No <a> tags found in the HTML file.")

except FileNotFoundError:
    print(f"Error: The file '{html_file_path}' was not found. Please ensure the path is correct.")
except Exception as e:
    print(f"An error occurred while parsing the HTML file: {e}")

The code above demonstrates how to search for `a`, `p`, and `div` tags. If you have specific HTML tags you'd like to investigate, please let me know, and I can modify the code to search for them.

In [ ]:
import razorpay

# --- IMPORTANT: Store your API keys securely, e.g., in Colab secrets ---
# For demonstration purposes, placeholders are used. Do not hardcode sensitive information.
# Example: RAZORPAY_KEY_ID = userdata.get('RAZORPAY_KEY_ID')
#          RAZORPAY_KEY_SECRET = userdata.get('RAZORPAY_KEY_SECRET')

RAZORPAY_KEY_ID = 'YOUR_RAZORPAY_KEY_ID' # Replace with your actual Key ID
RAZORPAY_KEY_SECRET = 'YOUR_RAZORPAY_KEY_SECRET' # Replace with your actual Key Secret

client = razorpay.Client(auth=(RAZORPAY_KEY_ID, RAZORPAY_KEY_SECRET))
client.set_app_details({"title": "Colab Notebook", "version": "1.0"})

print("RazorPay client initialized successfully!")
# You can test the client by fetching a list of payments (requires appropriate permissions):
# try:
#     payments = client.payment.all({'count': 1})
#     print(f"Successfully fetched {len(payments['items'])} payments.")
# except Exception as e:
#     print(f"Error fetching payments: {e}")

In [ ]:
%pip install razorpay

In [ ]:
import razorpay

# --- IMPORTANT: Store your API keys securely, e.g., in Colab secrets ---
# For demonstration purposes, placeholders are used. Do not hardcode sensitive information.
# Example: RAZORPAY_KEY_ID = userdata.get('RAZORPAY_KEY_ID')
#          RAZORPAY_KEY_SECRET = userdata.get('RAZORPAY_KEY_SECRET')

RAZORPAY_KEY_ID = 'YOUR_RAZORPAY_KEY_ID' # Replace with your actual Key ID
RAZORPAY_KEY_SECRET = 'YOUR_RAZORPAY_KEY_SECRET' # Replace with your actual Key Secret

client = razorpay.Client(auth=(RAZORPAY_KEY_ID, RAZORPAY_KEY_SECRET))
client.set_app_details({"title": "Colab Notebook", "version": "1.0"})

# Create a payment order
# Amount in paise (e.g., 10000 = 100.00 INR)
order_amount = 50000  # Example: 500 INR
order_receipt = 'receipt_1'

try:
    order = client.order.create({
        'amount': order_amount,
        'currency': 'INR',
        'receipt': order_receipt,
        'payment_capture': 1  # Auto capture payment
    })
    print("Payment order created successfully!")
    print(order)
except Exception as e:
    print(f"Error creating payment order: {e}")

## Wise API Integration

To interact with the Wise (formerly TransferWise) API, you will need to set up an API key. For testing purposes, you can use the sandbox environment provided by Wise. Make sure to store your API token securely, for example, in Colab secrets.

First, let's install the `wise` Python library.

In [ ]:
%pip install wise

Now, let's initialize the Wise client using your API token and demonstrate how to create a transfer. This example will use placeholder values for a test transaction.

In [ ]:
from wise.client import Client
from wise.constants import Currency, ProfileType

# --- IMPORTANT: Store your API keys securely, e.g., in Colab secrets ---
# For demonstration purposes, placeholders are used. Do not hardcode sensitive information.
# Example: WISE_API_TOKEN = userdata.get('WISE_API_TOKEN')

WISE_API_TOKEN = 'YOUR_WISE_API_TOKEN' # Replace with your actual Wise API token

# Initialize the Wise client
client = Client(private_token=WISE_API_TOKEN, sandbox=True) # Use sandbox=True for testing

print("Wise client initialized successfully!")

# Example: Get your profiles
try:
    profiles = client.get_profiles()
    personal_profile_id = None
    for profile in profiles:
        if profile['type'] == ProfileType.PERSONAL:
            personal_profile_id = profile['id']
            break

    if personal_profile_id:
        print(f"Found personal profile ID: {personal_profile_id}")
        # You can now proceed with creating quotes and transfers using this profile_id
    else:
        print("No personal profile found.")

except Exception as e:
    print(f"Error initializing Wise client or fetching profiles: {e}")


To perform an actual test transaction, you would typically follow these steps:

1.  **Get a Quote:** Determine the exchange rate and fees for your desired transfer.
2.  **Create a Recipient Account:** Specify where the money is going.
3.  **Create a Transfer:** Initiate the transfer based on the quote and recipient.
4.  **Fund the Transfer:** Depending on the payment method, you might need to approve or fund the transfer outside the API.

Here's a simplified example of getting a quote and attempting to create a transfer. **Note: This will likely fail without a valid recipient account and sufficient balance/funding in the sandbox environment, but it demonstrates the API calls.**

In [ ]:
# --- Wise Test Transaction Example ---

# Ensure the client is initialized and personal_profile_id is available
if 'client' in locals() and 'personal_profile_id' in locals() and personal_profile_id:
    try:
        # 1. Get a Quote
        # Replace with your actual currencies and amount
        source_currency = Currency.USD
        target_currency = Currency.EUR
        send_amount = 100.0 # Amount to send

        quote = client.create_quote(
            profile_id=personal_profile_id,
            source_currency=source_currency,
            target_currency=target_currency,
            send_amount=send_amount
        )
        print("\n--- Wise Quote Created ---")
        print(quote)

        # In a real scenario, you would first create a recipient account.
        # For this example, we'll skip recipient creation and directly try to create a transfer.
        # This will likely fail in sandbox without a valid recipient.

        # 2. (Skipping) Create a Recipient Account - This step is crucial in a real scenario
        # Example of how you would create a recipient (requires bank details):
        # recipient = client.create_account(
        #     profile_id=personal_profile_id,
        #     currency=target_currency,
        #     type="iban", # or "bank_code", etc.
        #     details={
        #         "accountHolderName": "Test Recipient",
        #         "iban": "GB...",
        #         "bic": "..."
        #     }
        # )
        # print("\n--- Recipient Account Created ---")
        # print(recipient)

        # 3. Create a Transfer (using the quote ID)
        # You would typically use the recipient['id'] here if you created one.
        # For this example, we're passing a placeholder, which will cause an error.
        # Always review Wise API documentation for required recipient details.
        test_target_account_id = 'YOUR_TEST_RECIPIENT_ACCOUNT_ID' # Placeholder, replace with actual recipient ID from client.create_account

        transfer = client.create_transfer(
            profile_id=personal_profile_id,
            quote_id=quote['id'],
            target_account_id=test_target_account_id,
            customer_transaction_id="my_unique_tx_id_123" # Unique ID for your reference
        )
        print("\n--- Wise Transfer Attempted ---")
        print(transfer)

    except Exception as e:
        print(f"\nError during Wise test transaction: {e}")
        print("Please ensure you have a valid personal profile ID and provide a valid 'target_account_id' for transfers in the sandbox environment.")
else:
    print("Wise client not initialized or personal profile ID not found. Cannot proceed with test transaction.")


## PayPal API Integration (Sandbox)

To interact with the PayPal API in a testing environment (Sandbox), you'll need to obtain Sandbox API credentials (Client ID and Client Secret) from your [PayPal Developer Dashboard](https://developer.paypal.com/).

**Important:** Store your Client ID and Client Secret securely, ideally in Colab secrets, and *never* hardcode them directly in your notebook for production environments.

First, let's install the `paypalrestsdk` Python library.

In [ ]:
%pip install paypalrestsdk

Now, let's initialize the PayPal API client using your Sandbox credentials. I'll include a small test to verify the connection.

In [ ]:
import paypalrestsdk
from google.colab import userdata # Assuming you store credentials in Colab secrets

# --- IMPORTANT: Store your API keys securely, e.g., in Colab secrets ---
# For demonstration purposes, placeholders are used. Do not hardcode sensitive information.
# Example: PAYPAL_CLIENT_ID = userdata.get('PAYPAL_CLIENT_ID')
#          PAYPAL_CLIENT_SECRET = userdata.get('PAYPAL_CLIENT_SECRET')

# Configure the PayPal SDK for Sandbox environment
paypalrestsdk.configure({
  "mode": "sandbox", # "sandbox" or "live"
  "client_id": PAYPAL_CLIENT_ID,
  "client_secret": PAYPAL_CLIENT_SECRET
})

print("PayPal SDK configured for Sandbox environment.")

# Test the API connection by trying to get an access token
try:
    # The SDK automatically handles token generation when making API calls
    # A simple way to verify configuration is to attempt a basic operation
    # For instance, listing payment definitions (if you have any for testing)
    # Or creating a simple payment object (which we will show in a separate cell)

    # For a direct token test, you'd typically make an OAuth call, but the SDK abstracts this.
    # Let's try to create a simple payment object to confirm setup.
    # This is just for setup verification, not a full transaction.
    payment = paypalrestsdk.Payment({
        "intent": "sale",
        "payer": {
            "payment_method": "paypal"
        },
        "transactions": [{
            "amount": {
                "total": "1.00",
                "currency": "USD"
            },
            "description": "Test payment for API initialization verification."
        }],
        "redirect_urls": {
            "return_url": "http://example.com/your_return_url",
            "cancel_url": "http://example.com/your_cancel_url"
        }
    })

    # This just creates the object, not the actual payment yet.
    # If the SDK setup is correct, this step should not raise an immediate configuration error.
    print("PayPal payment object created successfully (configuration seems valid).")
    print("To actually create a payment, you'd call `payment.create()` and handle redirects.")

except paypalrestsdk.exceptions.ConnectionError as e:
    print(f"Error connecting to PayPal API: {e}")
    print("Please check your network connection or PayPal API status.")


### Initiating and Executing a PayPal Payment (Sandbox)

Now that the PayPal SDK is configured, we can proceed with a full payment flow. This involves:

1.  **Creating a Payment:** Define the payment details (amount, currency, items, etc.).
2.  **Getting Approval URL:** PayPal returns a URL where the payer needs to approve the transaction.
3.  **User Redirection (Manual in Colab):** In a real web application, you would redirect the user's browser to this approval URL. In Colab, you'll need to manually open it.
4.  **Executing the Payment:** After the user approves, PayPal redirects them back to your `return_url`. You then use the `paymentId` and `PayerID` from the URL to execute the payment.

After opening the `Approval URL` from the previous cell in your browser, log in with your PayPal Sandbox buyer account and approve the payment. You will be redirected to `http://example.com/execute?paymentId=PAYMENT_ID&token=TOKEN&PayerID=PAYER_ID`.

**Copy the `paymentId` and `PayerID` values from that redirected URL and paste them into the variables below**, then run the cell to execute the payment.

In [ ]:
%pip install -U -q "google-genai>=2.9.0"


In [ ]:
# --- Step 3 & 4: Execute the Payment ---

# IMPORTANT: Replace these with the actual values from your PayPal redirect URL
payment_id_from_url = 'PASTE_PAYMENT_ID_HERE' # e.g., 'PAY-123ABCDEF456GHIJKL789MNO'
payer_id_from_url = 'PASTE_PAYER_ID_HERE'   # e.g., 'ABCDEFGHIJKL'

if payment_id_from_url == 'PASTE_PAYMENT_ID_HERE' or payer_id_from_url == 'PASTE_PAYER_ID_HERE':
    print("Please replace 'PASTE_PAYMENT_ID_HERE' and 'PASTE_PAYER_ID_HERE' with actual values from the PayPal redirect URL after approval.")
else:
    try:
        # Retrieve the payment object using the payment ID
        payment_to_execute = paypalrestsdk.Payment.find(payment_id_from_url)

        if payment_to_execute.execute({"payer_id": payer_id_from_url}):
            print("\nPayment executed successfully!")
            print(f"Payment State: {payment_to_execute.state}")
            print(f"Payment ID: {payment_to_execute.id}")
        else:
            print("\nError executing payment:")
            print(payment_to_execute.error)

    except Exception as e:
        print(f"An error occurred during payment execution: {e}")


In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
import uuid
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=GEMINI_API_KEY)

# The default managed agent.
AGENT = "antigravity-preview-05-2026"

# Generate a unique suffix for this notebook session to prevent agent ID conflicts
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

print("Client ready!")

In [ ]:
import uuid
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=GEMINI_API_KEY)

# The default managed agent.
AGENT = "antigravity-preview-05-2026"

# Generate a unique suffix for this notebook session to prevent agent ID conflicts
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

print("Client ready!")

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What is the capital of France?",
    environment="remote",
)

Markdown(interaction.output_text)

In [ ]:
# The response also includes metadata about the agent's sandbox.
print(f"Status:         {interaction.status}")
print(f"Interaction ID: {interaction.id}")
print(f"Environment ID: {interaction.environment_id}")

In [ ]:
# Turn 1: Introduce yourself.
turn1 = client.interactions.create(
    agent=AGENT,
    input="Hi! My name is Alice and I'm a software engineer. Remember that in a knowledge.md doc.",
    environment= "remote",
)

Markdown(f"**Turn 1:** {turn1.output_text}")

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Write a Python script that computes the first 20 Fibonacci numbers. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

In [ ]:

# Inspect the steps from the Fibonacci interaction above.
for i, step in enumerate(interaction.steps):
    step_type = step.type
    print(f"--- Step {i} [{step_type}] ---")

    # Tool call steps show which tool was invoked and with what arguments.
    if hasattr(step, "name") and step.name:
        print(f"  Tool: {step.name}")
        if hasattr(step, "arguments"):
            args_str = str(step.arguments)[:300]
            print(f"  Args: {args_str}")

    # Content steps contain the agent's text output.
    if hasattr(step, "content") and step.content:
        for c in step.content:
            if hasattr(c, "text"):
                print(f"  Text: {c.text[:300]}")
    print()

In [ ]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What were the top 3 news stories about Google this week? Summarize them briefly.",
    environment="remote",
)

Markdown(interaction.output_text)

In [ ]:
# Ask the agent to create a file, run it, and show results.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a Python file called 'analysis.py' that generates 50 random numbers, "
        "computes mean, median, and standard deviation, then prints the results. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

In [ ]:
import random
import statistics

# Set seed for reproducibility (optional, but good for consistency)
random.seed(42)

# Generate 50 random numbers between 1 and 100
numbers = [random.uniform(1, 100) for _ in range(50)]

# Calculate statistics
mean_val = statistics.mean(numbers)
median_val = statistics.median(numbers)
stdev_val = statistics.stdev(numbers)

# Print results
print("Generated 50 random numbers:")
print([round(num, 2) for num in numbers])
print(f"\nMean: {mean_val:.4f}")
print(f"Median: {median_val:.4f}")
print(f"Standard Deviation: {stdev_val:.4f}")

In [ ]:
# Inject a CSV file inline and ask the agent to analyze it.
csv_data = """name,age,city,score
              Alice,28,Paris,92
              Bob,35,London,87
              Charlie,42,Berlin,95
              Diana,31,Tokyo,88
              Eve,26,Sydney,91"""

interaction = client.interactions.create(
    agent=AGENT,
    input="Read the file data.csv, analyze it, and tell me who scored the highest.",
    environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "content": csv_data,
                "target": "/workspace/data.csv",
            }
        ],
    },
)

Markdown(interaction.output_text)

In [ ]:
import paypalrestsdk
from google.colab import userdata # Assuming you store credentials in Colab secrets

# --- IMPORTANT: Store your API keys securely, e.g., in Colab secrets ---
# For demonstration purposes, placeholders are used. Do not hardcode sensitive information.
# Example: PAYPAL_CLIENT_ID = userdata.get('PAYPAL_CLIENT_ID')
#          PAYPAL_CLIENT_SECRET = userdata.get('PAYPAL_CLIENT_SECRET')

# Replace with your actual PayPal Sandbox Client ID and Client Secret
PAYPAL_CLIENT_ID = 'YOUR_PAYPAL_SANDBOX_CLIENT_ID'
PAYPAL_CLIENT_SECRET = 'YOUR_PAYPAL_SANDBOX_CLIENT_SECRET'

# Configure the PayPal SDK for Sandbox environment
paypalrestsdk.configure({
  "mode": "sandbox", # "sandbox" or "live"
  "client_id": PAYPAL_CLIENT_ID,
  "client_secret": PAYPAL_CLIENT_SECRET
})

print("PayPal SDK configured for Sandbox environment.")

# Test the API connection by trying to get an access token
try:
    # The SDK automatically handles token generation when making API calls
    # A simple way to verify configuration is to attempt a basic operation
    # For instance, listing payment definitions (if you have any for testing)
    # Or creating a simple payment object (which we will show in a separate cell)

    # For a direct token test, you'd typically make an OAuth call, but the SDK abstracts this.
    # Let's try to create a simple payment object to confirm setup.
    # This is just for setup verification, not a full transaction.
    payment = paypalrestsdk.Payment({
        "intent": "sale",
        "payer": {
            "payment_method": "paypal"
        },
        "transactions": [{
            "amount": {
                "total": "1.00",
                "currency": "USD"
            },
            "description": "Test payment for API initialization verification."
        }],
        "redirect_urls": {
            "return_url": "http://example.com/your_return_url",
            "cancel_url": "http://example.com/your_cancel_url"
        }
    })

    # This just creates the object, not the actual payment yet.
    # If the SDK setup is correct, this step should not raise an immediate configuration error.
    print("PayPal payment object created successfully (configuration seems valid).")
    print("To actually create a payment, you'd call `payment.create()` and handle redirects.")

except paypalrestsdk.exceptions.ConnectionError as e:
    print(f"Error connecting to PayPal API: {e}")
    print("Please check your network connection or PayPal API status.")
except Exception as e:
    print(f"An unexpected error occurred during PayPal SDK configuration test: {e}")
    print("Ensure your Client ID and Client Secret are correct and have appropriate permissions.")

In [ ]:
%pip install ngrok

Once you have added your `NGROK_AUTH_TOKEN` to Colab secrets, run the following cell to authenticate ngrok. Remember to enable 'Notebook access' for the secret.

In [ ]:
from google.colab import userdata
import ngrok

# Get the ngrok authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Authenticate ngrok
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authtoken set successfully.")
else:
    print("NGROK_AUTH_TOKEN not found in Colab secrets. Please add it.")

In [121]:
from google.colab import userdata
import ngrok

# Get the ngrok authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Authenticate ngrok
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authtoken set successfully.")
else:
    print("NGROK_AUTH_TOKEN not found in Colab secrets. Please add it.")

ngrok authtoken set successfully.


In [120]:
import asyncio

# The port your local server is running on
local_port = 5000

async def start_ngrok_tunnel(port):
    try:
        # Establish a tunnel to your local server
        public_url = await ngrok.connect(port)
        print(f"ngrok tunnel established at: {public_url}")
        print(f"Forwarding to http://localhost:{port}")
    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")
        print("Please ensure your local server is running on the specified port and your authtoken is valid.")

# Run the async function
# This needs to be run in an event loop, which Colab handles implicitly for top-level await
# If you encounter issues, you might need to explicitly manage the loop:
# asyncio.run(start_ngrok_tunnel(local_port))

# In Colab, you can often just call the async function directly.
await start_ngrok_tunnel(local_port)

ngrok tunnel established at: <builtins.Listener object at 0x7abf939216b0>
Forwarding to http://localhost:5000


In [119]:
import ngrok

print("Attempting to kill existing ngrok processes...")
ngrok.kill()
print("ngrok processes killed (if any were running).")

Attempting to kill existing ngrok processes...
ngrok processes killed (if any were running).


In [118]:
from google.colab import userdata
import ngrok

# Get the ngrok authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Authenticate ngrok
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authtoken set successfully.")
else:
    print("NGROK_AUTH_TOKEN not found in Colab secrets. Please add it.")

ngrok authtoken set successfully.


In [117]:
import asyncio

# The port your local server is running on
local_port = 5000

async def start_ngrok_tunnel(port):
    try:
        # Establish a tunnel to your local server
        public_url = await ngrok.connect(port)
        print(f"ngrok tunnel established at: {public_url}")
        print(f"Forwarding to http://localhost:{port}")
    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")
        print("Please ensure your local server is running on the specified port and your authtoken is valid.")

# Run the async function
# This needs to be run in an event loop, which Colab handles implicitly for top-level await
# If you encounter issues, you might need to explicitly manage the loop:
# asyncio.run(start_ngrok_tunnel(local_port))

# In Colab, you can often just call the async function directly.
await start_ngrok_tunnel(local_port)

ngrok tunnel established at: <builtins.Listener object at 0x7abf91d8fb70>
Forwarding to http://localhost:5000


In [ ]:
ngrok is already installed in this notebook. The persistent `ERR_NGROK_334` (endpoint already online) error requires a full Colab runtime restart.

**Please restart the Colab runtime (Runtime -> Restart runtime).**

After restarting, re-run the `ngrok` authentication cell (`bb45cad9` or `edea79b3`) and then the `ngrok` tunnel creation cell (`f987549f` or `caef52cc`).

Now, let's start the ngrok tunnel. By default, it will connect to port `5000`. If your local server is running on a different port, please modify the `local_port` variable below.

In [123]:
import asyncio
import ngrok

# The port your local server is running on
local_port = 5000
global_ngrok_public_url = None # Initialize global variable

async def start_ngrok_tunnel(port):
    global global_ngrok_public_url # Declare intent to modify global variable
    try:
        # Establish a tunnel to your local server
        public_url = await ngrok.connect(port)
        global_ngrok_public_url = public_url # Assign to global variable
        print(f"ngrok tunnel established at: {public_url}")
        print(f"Forwarding to http://localhost:{port}")
    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")
        print("Please ensure your local server is running on the specified port and your authtoken is valid.")

# In Colab, you can often just call the async function directly.
await start_ngrok_tunnel(local_port)

ngrok tunnel established at: <builtins.Listener object at 0x7abf904c6d30>
Forwarding to http://localhost:5000


In [139]:
import subprocess
import os
import time

# Path to your backend application
backend_app_path = '/content/backend_app.py'

# Define the correct default content for backend_app.py
default_app_content = '''
import os
from flask import Flask, request, jsonify
import stripe

app = Flask(__name__)

# Configure Stripe - Prioritize Colab Secrets/Environment Variables
api_key = os.environ.get('STRIPE_SECRET_KEY', 'sk_test_placeholder')
stripe.api_key = api_key

# Default route for ngrok verification
@app.route('/', methods=['GET'])
def home():
    return jsonify(message='Hello from Colab backend!'), 200

# Existing Stripe route
@app.route('/api/create-stripe-session', methods=['POST'])
def create_checkout_session():
    if stripe.api_key == 'sk_test_placeholder':
        return jsonify(error='Stripe API Key not configured. Please add STRIPE_SECRET_KEY to Colab Secrets.'), 401

    try:
        data = request.json
        # Stripe India requires specific handling for currency and compliance
        session = stripe.checkout.Session.create(
            payment_method_types=['card'],
            line_items=[{
                'price_data': {
                    'currency': 'inr', # Mandatory for domestic Indian transactions
                    'product_data': {'name': data.get('item_name', 'Talkie Subscription')},
                    'unit_amount': int(float(data.get('amount', '500')) * 100),
                },
                'quantity': 1,
            }],
            mode='payment',
            success_url='http://localhost:5000/success',
            cancel_url='http://localhost:5000/cancel',
            # Adding a description helps with Indian regulatory compliance (GST/Export)
            payment_intent_data={
                'description': 'Talkie Talkie Software Subscription Service',
            }
        )
        return jsonify({'id': session.id, 'url': session.url})
    except stripe.error.AuthenticationError:
        return jsonify(error='Invalid or Expired Stripe API Key.'), 401
    except Exception as e:
        return jsonify(error=str(e)), 403

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
'''

# Always write the corrected content to backend_app.py
print(f"Writing corrected content to '{backend_app_path}'.")
with open(backend_app_path, 'w') as f:
    f.write(default_app_content)

print(f"Starting backend_app.py on port 5000 in the background...")

# Use nohup to run in the background and redirect output to a log file
try:
    # Kill any process currently listening on port 5000
    subprocess.run(['fuser', '-k', '5000/tcp'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1) # Give it a moment to clear

    command = f"nohup python3 {backend_app_path} > server.log 2>&1 &"
    subprocess.run(command, shell=True, check=True)
    print("backend_app.py started. Check server.log for output.")
    # Give it a moment to start up
    time.sleep(5)
except subprocess.CalledProcessError as e:
    print(f"Failed to start backend_app.py: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Writing corrected content to '/content/backend_app.py'.
Starting backend_app.py on port 5000 in the background...
backend_app.py started. Check server.log for output.


In [129]:
# Re-run the ngrok verification after starting the local server
# The global_ngrok_public_url variable is set in cell caef52cc
if 'global_ngrok_public_url' in globals() and global_ngrok_public_url is not None:
    print("Re-verifying ngrok tunnel after starting local server...")
    await verify_ngrok_tunnel(global_ngrok_public_url)
else:
    print("Error: 'global_ngrok_public_url' not found or is None. Please ensure the ngrok tunnel was started successfully in cell caef52cc.")

Re-verifying ngrok tunnel after starting local server...
Attempting to reach ngrok tunnel at: https://satisfied-handbook-lizard.ngrok-free.dev
Status Code from ngrok tunnel: 404
ngrok tunnel returned a non-200 status code: 404.
This might indicate an issue with the local server or its configuration.


In [136]:
import os

log_file_path = '/content/server.log'

if os.path.exists(log_file_path):
    with open(log_file_path, 'r') as f:
        log_content = f.read()
    if log_content:
        print("Content of server.log:")
        print(log_content)
    else:
        print(f"The server.log file at '{log_file_path}' is empty. This could mean the backend application didn't start or log any output.")
else:
    print(f"Error: The server.log file was not found at '{log_file_path}'.")

Content of server.log:
 * Serving Flask app 'backend_app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit
127.0.0.1 - - [14/Aug/2026 07:23:35] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [14/Aug/2026 07:27:24] "GET / HTTP/1.1" 404 -



In [138]:
import os

backend_app_path = '/content/backend_app.py'

if os.path.exists(backend_app_path):
    with open(backend_app_path, 'r') as f:
        content = f.read()
    print(content)
else:
    print(f"Error: The file '{backend_app_path}' was not found.")


import os
from flask import Flask, request, jsonify
import stripe

app = Flask(__name__)

# Configure Stripe - Prioritize Colab Secrets/Environment Variables
api_key = os.environ.get('STRIPE_SECRET_KEY', 'sk_test_placeholder')
stripe.api_key = api_key

@app.route('/api/create-stripe-session', methods=['POST'])
def create_checkout_session():
    if stripe.api_key == 'sk_test_placeholder':
        return jsonify(error='Stripe API Key not configured. Please add STRIPE_SECRET_KEY to Colab Secrets.'), 401
        
    try:
        data = request.json
        # Stripe India requires specific handling for currency and compliance
        session = stripe.checkout.Session.create(
            payment_method_types=['card'],
            line_items=[{
                'price_data': {
                    'currency': 'inr', # Mandatory for domestic Indian transactions
                    'product_data': {'name': data.get('item_name', 'Talkie Subscription')},
                    'unit_amount': int(flo

In [140]:
import subprocess

print("Attempting to curl http://localhost:5000 directly...")
try:
    # Use -m to limit time, -s for silent output (no progress bar), -o /dev/null for discarding output
    # and -w '%{http_code}' to get only the HTTP status code
    result = subprocess.run(['curl', '-s', '-o', '/dev/null', '-w', '%{http_code}', 'http://localhost:5000'], capture_output=True, text=True, timeout=10)
    http_code = result.stdout.strip()
    print(f"Curl status code: {http_code}")

    # Now, get the actual content of the root path to confirm the message
    content_result = subprocess.run(['curl', '-s', 'http://localhost:5000'], capture_output=True, text=True, timeout=10)
    print("Curl response body:")
    print(content_result.stdout.strip())

    if http_code == '200' and 'Hello from Colab backend!' in content_result.stdout:
        print("\nLocal Flask server is running and accessible (Status 200 OK)!")
    else:
        print("\nLocal Flask server is NOT returning 200 OK or the expected message. It might not be running or is misconfigured.")
        if result.stderr:
            print("Curl error output:", result.stderr)

except subprocess.TimeoutExpired:
    print("Curl command timed out. Local Flask server might not be running or is unresponsive.")
except Exception as e:
    print(f"An error occurred while curling the local server: {e}")
    print("This might indicate the Flask server is not running or there's a networking issue within Colab.")


Attempting to curl http://localhost:5000 directly...
Curl status code: 200
Curl response body:
{"message":"Hello from Colab backend!"}

Local Flask server is running and accessible (Status 200 OK)!


In [127]:
import requests
import asyncio

async def verify_ngrok_tunnel(public_url_obj):
    try:
        # Access the URL from the Listener object by calling its .url() method
        public_url_str = public_url_obj.url()
        print(f"Attempting to reach ngrok tunnel at: {public_url_str}")

        response = requests.get(public_url_str)
        print(f"Status Code from ngrok tunnel: {response.status_code}")
        if response.status_code == 200:
            print("ngrok tunnel is successfully forwarding requests (status 200 OK).")
        else:
            print(f"ngrok tunnel returned a non-200 status code: {response.status_code}.")
            print("This might indicate an issue with the local server or its configuration.")
    except Exception as e:
        print(f"Error verifying ngrok tunnel: {e}")
        print("Make sure your local server is running on the specified port (5000 by default) and the ngrok tunnel is active.")

# Run the async verification function, passing the global_ngrok_public_url object if it exists
# The global_ngrok_public_url variable is set in cell caef52cc
if 'global_ngrok_public_url' in globals() and global_ngrok_public_url is not None:
    await verify_ngrok_tunnel(global_ngrok_public_url)
else:
    print("Error: 'global_ngrok_public_url' not found or is None. Please ensure the ngrok tunnel was started successfully in cell caef52cc.")

Attempting to reach ngrok tunnel at: https://satisfied-handbook-lizard.ngrok-free.dev
Status Code from ngrok tunnel: 502
ngrok tunnel returned a non-200 status code: 502.
This might indicate an issue with the local server or its configuration.


In [132]:
pip install ngrok

Once you have added your `NGROK_AUTH_TOKEN` to Colab secrets, run the following cell to authenticate ngrok. Remember to enable 'Notebook access' for the secret.

In [133]:
from google.colab import userdata
import ngrok

# Get the ngrok authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Authenticate ngrok
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authtoken set successfully.")
else:
    print("NGROK_AUTH_TOKEN not found in Colab secrets. Please add it.")

ngrok authtoken set successfully.


Now, let's start the ngrok tunnel. By default, it will connect to port `5000`. If your local server is running on a different port, please modify the `local_port` variable below.

In [134]:
import asyncio
import ngrok

# The port your local server is running on
local_port = 5000
global_ngrok_public_url = None # Initialize global variable

async def start_ngrok_tunnel(port):
    global global_ngrok_public_url # Declare intent to modify global variable
    try:
        # Establish a tunnel to your local server
        public_url = await ngrok.connect(port)
        global_ngrok_public_url = public_url # Assign to global variable
        print(f"ngrok tunnel established at: {public_url}")
        print(f"Forwarding to http://localhost:{port}")
    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")
        print("Please ensure your local server is running on the specified port and your authtoken is valid.")

# In Colab, you can often just call the async function directly.
await start_ngrok_tunnel(local_port)

ngrok tunnel established at: <builtins.Listener object at 0x7abf91f515b0>
Forwarding to http://localhost:5000


In [135]:
import requests
import asyncio

async def verify_ngrok_tunnel(public_url_obj):
    try:
        # Access the URL from the Listener object by calling its .url() method
        public_url_str = public_url_obj.url()
        print(f"Attempting to reach ngrok tunnel at: {public_url_str}")

        response = requests.get(public_url_str)
        print(f"Status Code from ngrok tunnel: {response.status_code}")
        if response.status_code == 200:
            print("ngrok tunnel is successfully forwarding requests (status 200 OK).")
        else:
            print(f"ngrok tunnel returned a non-200 status code: {response.status_code}.")
            print("This might indicate an issue with the local server or its configuration.")
    except Exception as e:
        print(f"Error verifying ngrok tunnel: {e}")
        print("Make sure your local server is running on the specified port (5000 by default) and the ngrok tunnel is active.")

# Run the async verification function, passing the global_ngrok_public_url object if it exists
# The global_ngrok_public_url variable is set in cell caef52cc
if 'global_ngrok_public_url' in globals() and global_ngrok_public_url is not None:
    await verify_ngrok_tunnel(global_ngrok_public_url)
else:
    print("Error: 'global_ngrok_public_url' not found or is None. Please ensure the ngrok tunnel was started successfully in cell caef52cc.")

Attempting to reach ngrok tunnel at: https://satisfied-handbook-lizard.ngrok-free.dev
Status Code from ngrok tunnel: 404
ngrok tunnel returned a non-200 status code: 404.
This might indicate an issue with the local server or its configuration.
